# ET 핵심 모듈 정량평가 실험 환경

이 노트북은 현재 프로젝트의 **신뢰도 점수 산출**과 **임베딩 기반 개인화 추천**을 같은 실행 기록 안에서 평가한다. 기본 실행은 외부 API/DB 없이 완료되며, Live 평가는 루트 `.env`의 스위치와 참조 경로로만 활성화한다. 비밀 키나 데이터 본문을 셀에 직접 입력하지 않는다.

- 기본 오프라인: 점수 산식 회귀검증, 기존 LLM 캐시 불변쌍 평가, 합성 추천 랭킹/후처리 검증
- `RUN_LIVE_TRUST=1`: `GEMINI_API_KEY`와 참조 JSONL을 사용한 실제 신뢰도 평가
- `RUN_DB_RECOMMENDATION=1`: `DATABASE_URL`의 이벤트 로그를 사용한 추천 리플레이 평가
- 마지막 셀: 표와 해석을 `EXPERIMENT_REPORT_DIR` 아래 Markdown/CSV로 저장


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import statistics
import subprocess
import sys
import time
import tracemalloc
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'backend' / 'services' / 'trust.py').exists():
            return candidate
    raise FileNotFoundError('backend/services/trust.py를 포함한 프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env', override=False)

def env_bool(name: str, default: bool = False) -> bool:
    return os.getenv(name, str(int(default))).strip().lower() in {'1', 'true', 'yes', 'on'}

def env_int(name: str, default: int) -> int:
    try:
        return int(os.getenv(name, str(default)))
    except ValueError:
        return default

def resolve_ref(env_name: str, default: str) -> Path:
    raw = Path(os.getenv(env_name, default))
    return raw if raw.is_absolute() else PROJECT_ROOT / raw

CONFIG = {
    'run_live_trust': env_bool('RUN_LIVE_TRUST'),
    'run_db_recommendation': env_bool('RUN_DB_RECOMMENDATION'),
    'trust_data_path': resolve_ref('TRUST_DATA_PATH', 'data/trust_eval_snu.jsonl'),
    'trust_pair_path': resolve_ref('TRUST_PAIR_PATH', 'data/invariant_pairs.jsonl'),
    'trust_cache_path': resolve_ref('TRUST_CACHE_PATH', 'FINAL_ANALYSIS/artifacts/optimization/trust_cache_evidence_v2.jsonl'),
    'trust_cache_overlay_path': resolve_ref('TRUST_CACHE_OVERLAY_PATH', 'FINAL_ANALYSIS/artifacts/optimization/trust_cache_logic_v3_p5.jsonl'),
    'trust_result_cache_path': resolve_ref('TRUST_RESULT_CACHE_PATH', 'FINAL_ANALYSIS/artifacts/optimization/trust_snu_w4_results.jsonl'),
    'trust_grounding_summary_path': resolve_ref('TRUST_GROUNDING_SUMMARY_PATH', 'FINAL_ANALYSIS/artifacts/optimization/trust_grounding_summary.json'),
    'report_dir': resolve_ref('EXPERIMENT_REPORT_DIR', 'FINAL_ANALYSIS/artifacts/core_function_experiment'),
    'max_live_samples': env_int('MAX_LIVE_SAMPLES', 10),
    'top_k': env_int('TOP_K', 3),
    'negative_count': env_int('NEGATIVE_COUNT', 3),
    'min_history': env_int('MIN_HISTORY', 1),
    'random_seed': env_int('RANDOM_SEED', 42),
}
CONFIG['report_dir'].mkdir(parents=True, exist_ok=True)

config_view = {
    **{k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()},
    'gemini_key_available': bool(os.getenv('GEMINI_API_KEY')),
    'database_url_available': bool(os.getenv('DATABASE_URL')),
}
pd.Series(config_view, name='value').to_frame()


,value
run_live_trust,False
run_db_recommendation,False
trust_data_path,C:\Users\user\Desktop\ET_by_claude\data\trust_...
trust_pair_path,C:\Users\user\Desktop\ET_by_claude\data\invari...
trust_cache_path,C:\Users\user\Desktop\ET_by_claude\FINAL_ANALY...
trust_cache_overlay_path,C:\Users\user\Desktop\ET_by_claude\FINAL_ANALY...
trust_result_cache_path,C:\Users\user\Desktop\ET_by_claude\FINAL_ANALY...
trust_grounding_summary_path,C:\Users\user\Desktop\ET_by_claude\FINAL_ANALY...
report_dir,C:\Users\user\Desktop\ET_by_claude\FINAL_ANALY...
max_live_samples,10


In [2]:
from backend.services import encoder_inference, recommend, trust
from backend.services.eval_trust import WEIGHT_VARIANTS
from backend.training.evaluate_offline import STRATEGIES, evaluate as evaluate_recommendation
from backend.training.samples import InteractionSample

def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    with path.open(encoding='utf-8') as file:
        for line_no, line in enumerate(file, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'{path.name}:{line_no} JSON 파싱 실패') from exc
    return rows

def percentile(values: list[float], q: float) -> float | None:
    if not values:
        return None
    ordered = sorted(values)
    index = min(len(ordered) - 1, max(0, math.ceil(q * len(ordered)) - 1))
    return float(ordered[index])

def git_value(*args: str) -> str:
    try:
        return subprocess.check_output(['git', *args], cwd=PROJECT_ROOT, text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return 'unknown'

evaluation_code_paths = [
    PROJECT_ROOT / 'backend/services/trust.py',
    PROJECT_ROOT / 'backend/services/recommend.py',
    PROJECT_ROOT / 'backend/services/eval_trust.py',
    PROJECT_ROOT / 'backend/training/evaluate_offline.py',
    PROJECT_ROOT / 'backend/training/samples.py',
    PROJECT_ROOT / 'backend/training/persona_benchmark.py',
]
code_digest = hashlib.sha256()
for code_path in evaluation_code_paths:
    code_digest.update(str(code_path.relative_to(PROJECT_ROOT)).encode('utf-8'))
    code_digest.update(code_path.read_bytes() if code_path.exists() else b'<missing>')
evaluation_code_sha256 = code_digest.hexdigest()
git_status_lines = [line for line in git_value('status', '--short').splitlines() if line.strip()]
worktree_dirty = bool(git_status_lines)

checkpoint = encoder_inference.DEFAULT_MODEL_PATH
runtime_df = pd.DataFrame([
    {'check': 'project_root', 'value': str(PROJECT_ROOT)},
    {'check': 'git_commit', 'value': git_value('rev-parse', '--short', 'HEAD')},
    {'check': 'worktree_dirty', 'value': worktree_dirty},
    {'check': 'changed_or_untracked_path_count', 'value': len(git_status_lines)},
    {'check': 'evaluation_code_sha256', 'value': evaluation_code_sha256},
    {'check': 'python', 'value': sys.version.split()[0]},
    {'check': 'python_executable', 'value': sys.executable},
    {'check': 'running_in_etvenv1', 'value': 'etvenv1' in sys.executable.lower()},
    {'check': 'trust_cache_modified_utc', 'value': datetime.fromtimestamp(CONFIG['trust_cache_path'].stat().st_mtime, timezone.utc).isoformat() if CONFIG['trust_cache_path'].exists() else 'missing'},
    {'check': 'attention_checkpoint_exists', 'value': checkpoint.exists()},
    {'check': 'attention_checkpoint_mb', 'value': round(checkpoint.stat().st_size / 1024**2, 2) if checkpoint.exists() else 0},
    {'check': 'attention_model_ready_here', 'value': encoder_inference.is_model_ready()},
    {'check': 'profile_embedding_dim', 'value': recommend.EMBED_DIM},
])
runtime_df


,check,value
0,project_root,C:\Users\user\Desktop\ET_by_claude
1,git_commit,1016f99d
2,worktree_dirty,True
3,changed_or_untracked_path_count,25
4,evaluation_code_sha256,62c1937d95fa7c7ef47e5d4cd605931908c9bb9ce12353...
5,python,3.11.9
6,python_executable,C:\Users\user\Desktop\ET_by_claude\etvenv1\Scr...
7,running_in_etvenv1,True
8,trust_cache_modified_utc,2026-07-11T01:44:04.224413+00:00
9,attention_checkpoint_exists,True


## A. 신뢰도 산출 — 결정론적 산식/경계 회귀검증

외부 LLM을 호출하기 전에 룰 점수, predicate 변환, 가중합, 교차 패널티, verdict 경계를 검증한다. 이 검증이 실패하면 Live 결과를 해석하지 말고 구현 회귀부터 수정해야 한다.


In [3]:
def criteria(source=10, evidence=10, style=10, logic=10, clickbait=0) -> dict:
    return {
        'source_credibility': {'score': source},
        'evidence_support': {'score': evidence},
        'style_neutrality': {'score': style},
        'logical_consistency': {'score': logic},
        'clickbait_risk': {'score': clickbait},
    }

unit_cases = [
    ('maximum_without_clickbait', trust._weighted_sum_score(criteria()), 100),
    ('verdict_true_boundary', trust._verdict(70), 'likely_true'),
    ('verdict_uncertain_boundary', trust._verdict(40), 'uncertain'),
    ('verdict_false_boundary', trust._verdict(39), 'likely_false'),
    ('known_source_rule', trust._rule_source_score('연합뉴스')['score'], 10),
    ('unknown_source_neutral', trust._rule_source_score('등록되지않은매체')['score'], 5),
    ('all_yes_predicate', trust._predicate_to_score([{'answer': 'yes'}] * 3)['score'], 10),
    ('all_uncertain_predicate', trust._predicate_to_score([{'answer': 'uncertain'}] * 3)['score'], 3),
]
cross_input = criteria(3, 3, 3, 3, 10)
cross_raw = sum(cross_input[key]['score'] * weight for key, weight in trust.WEIGHTS.items())
cross_base = int((cross_raw / trust._MAX_RAW) * 100)
unit_cases.append(('cross_penalty_and_evidence_cap', trust._weighted_sum_score(cross_input), min(69, max(0, cross_base - 10))))
trust_unit_df = pd.DataFrame(unit_cases, columns=['test', 'actual', 'expected'])
trust_unit_df['pass'] = trust_unit_df['actual'] == trust_unit_df['expected']
assert trust_unit_df['pass'].all(), trust_unit_df.loc[~trust_unit_df['pass']]
trust_unit_df


,test,actual,expected,pass
0,maximum_without_clickbait,100,100,True
1,verdict_true_boundary,likely_true,likely_true,True
2,verdict_uncertain_boundary,uncertain,uncertain,True
3,verdict_false_boundary,likely_false,likely_false,True
4,known_source_rule,10,10,True
5,unknown_source_neutral,5,5,True
6,all_yes_predicate,10,10,True
7,all_uncertain_predicate,3,3,True
8,cross_penalty_and_evidence_cap,0,0,True


## B. 신뢰도 산출 — 기존 캐시 불변쌍 및 가중치 민감도

최신 optimization 캐시의 기준별 LLM 판정을 재사용하므로 API 호출 없이 가중치 조합별 방향성 통과율과 margin을 비교한다. P5 재평가 overlay를 pair_id+side 기준으로 합친다.


In [4]:
def apply_weights(per_criteria: dict, weights: dict) -> int:
    max_raw = sum(weight for weight in weights.values() if weight > 0) * 10
    raw = sum(per_criteria[key]['score'] * weight for key, weight in weights.items())
    score = (raw / max_raw) * 100
    if per_criteria['style_neutrality']['score'] < 4 and per_criteria['evidence_support']['score'] < 4:
        score -= 10
    if per_criteria['evidence_support']['score'] < 4:
        score = min(score, 69)
    return int(max(0, min(100, score)))

pairs = load_jsonl(CONFIG['trust_pair_path'])
cache_rows = load_jsonl(CONFIG['trust_cache_path'])
overlay_rows = load_jsonl(CONFIG['trust_cache_overlay_path'])
if overlay_rows:
    merged_cache = {(row['pair_id'], row['side']): row for row in cache_rows}
    merged_cache.update({(row['pair_id'], row['side']): row for row in overlay_rows})
    cache_rows = list(merged_cache.values())
cache_index = {(row['pair_id'], row['side']): row for row in cache_rows}
pair_results = []
for weight_name, weights in WEIGHT_VARIANTS.items():
    for pair in pairs:
        pair_id = pair['pair_id']
        record_a = cache_index.get((pair_id, 'a'))
        if not record_a:
            continue
        score_a = apply_weights(record_a['per_criteria'], weights)
        if pair['type'] == 'single':
            low, high = pair.get('expected_range', [0, 100])
            passed = low <= score_a <= high
            margin = min(score_a - low, high - score_a)
            score_b = None
        else:
            record_b = cache_index.get((pair_id, 'a_prime'))
            if not record_b:
                continue
            score_b = apply_weights(record_b['per_criteria'], weights)
            signed = score_a - score_b if pair.get('expected') == 'a_gt_a_prime' else score_b - score_a
            passed = signed > 0
            margin = signed
        pair_results.append({
            'weights': weight_name, 'pair_id': pair_id, 'case_type': pair['type'],
            'signal': pair.get('signal', ''), 'score_a': score_a, 'score_a_prime': score_b,
            'margin': margin, 'pass': passed,
        })

trust_pair_df = pd.DataFrame(pair_results)
if trust_pair_df.empty:
    trust_pair_summary_df = pd.DataFrame([{'status': 'cache_or_pair_file_missing'}])
else:
    current_weight_name = next((name for name, weights in WEIGHT_VARIANTS.items() if weights == trust.WEIGHTS), 'custom')
    summary_rows = []
    for weight_name, group in trust_pair_df.groupby('weights', sort=False):
        pair_only = group[group['case_type'] == 'pair']
        summary_rows.append({
            'weights': weight_name, 'is_current': weight_name == current_weight_name,
            'cases': len(group), 'passed': int(group['pass'].sum()), 'pass_rate': float(group['pass'].mean()),
            'failed_cases': ', '.join(group.loc[~group['pass'], 'pair_id'].astype(str)) or '-',
            'mean_pair_margin': float(pair_only['margin'].mean()) if not pair_only.empty else None,
            'min_pair_margin': float(pair_only['margin'].min()) if not pair_only.empty else None,
        })
    trust_pair_summary_df = pd.DataFrame(summary_rows).sort_values(['is_current', 'weights'], ascending=[False, True])
trust_pair_summary_df


,weights,is_current,cases,passed,pass_rate,failed_cases,mean_pair_margin,min_pair_margin
2,W2,True,9,9,1.0,-,30.500,5.0
0,W0,False,9,9,1.0,-,29.000,6.0
1,W1,False,9,9,1.0,-,29.750,5.0
3,W3,False,9,9,1.0,-,28.875,6.0
4,W4,False,9,9,1.0,-,29.500,3.0


## C. 신뢰도 산출 — 선택적 Gemini Live 평가

`RUN_LIVE_TRUST=1`일 때만 실행한다. `MAX_LIVE_SAMPLES`만큼 참조 JSONL을 읽고 정확도·정밀도·재현율·F1·ROC-AUC·Spearman·실패율·p50/p95 지연시간을 계산한다. 실패(`score=None`)는 0점으로 바꾸지 않고 별도로 센다.


In [5]:
def binary_metrics(frame: pd.DataFrame, threshold: int = 70) -> dict:
    valid = frame.dropna(subset=['score', 'label']).copy()
    if valid.empty:
        return {'n_valid': 0}
    spearman = valid['score'].rank(method='average').corr(valid['label'].rank(method='average'))
    extreme = valid[(valid['label'] <= 0.25) | (valid['label'] >= 0.75)].copy()
    if extreme.empty:
        return {'n_valid': len(valid), 'n_extreme': 0, 'spearman_rho': spearman}
    y = (extreme['label'] >= 0.75).astype(int)
    pred = (extreme['score'] >= threshold).astype(int)
    tp = int(((y == 1) & (pred == 1)).sum())
    tn = int(((y == 0) & (pred == 0)).sum())
    fp = int(((y == 0) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    ranks = extreme['score'].rank(method='average')
    n_pos, n_neg = int((y == 1).sum()), int((y == 0).sum())
    auc = ((ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)) if n_pos and n_neg else None
    return {
        'n_valid': len(valid), 'n_extreme': len(extreme), 'accuracy': (tp + tn) / len(extreme), 'precision': precision,
        'recall': recall, 'f1': f1, 'roc_auc': auc, 'spearman_rho': spearman,
        'high_label_mean': float(extreme.loc[y == 1, 'score'].mean()),
        'low_label_mean': float(extreme.loc[y == 0, 'score'].mean()),
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
    }

live_rows = []
live_error = None
if CONFIG['run_live_trust']:
    if not os.getenv('GEMINI_API_KEY'):
        live_error = 'RUN_LIVE_TRUST=1이지만 GEMINI_API_KEY가 없습니다.'
    else:
        samples = load_jsonl(CONFIG['trust_data_path'])[:CONFIG['max_live_samples']]
        for index, sample in enumerate(samples, 1):
            started = time.perf_counter()
            try:
                result = trust.score_trust(sample.get('text', ''), sample.get('source'), sample.get('title'))
                live_rows.append({
                    'sample_no': index, 'label': sample.get('label'), 'score': result.get('score'),
                    'verdict': result.get('verdict'), 'latency_sec': time.perf_counter() - started,
                    'failed': result.get('score') is None,
                })
            except Exception as exc:
                live_rows.append({'sample_no': index, 'label': sample.get('label'), 'score': None,
                                  'verdict': 'exception', 'latency_sec': time.perf_counter() - started, 'failed': True})
                live_error = f'{type(exc).__name__}: {exc}'

if not live_rows and CONFIG['trust_result_cache_path'].exists():
    for index, cached in enumerate(load_jsonl(CONFIG['trust_result_cache_path']), 1):
        score = apply_weights(cached['per_criteria'], trust.WEIGHTS)
        live_rows.append({
            'sample_no': index, 'label': cached.get('label'), 'score': score,
            'verdict': trust._verdict(score), 'latency_sec': None, 'failed': False, 'source_mode': 'cached_live',
        })

trust_live_df = pd.DataFrame(live_rows)
if trust_live_df.empty:
    trust_live_summary_df = pd.DataFrame([{'status': live_error or 'skipped_by_RUN_LIVE_TRUST'}])
else:
    metrics = binary_metrics(trust_live_df)
    metrics.update({
        'requested': len(trust_live_df), 'failure_rate': float(trust_live_df['failed'].mean()),
        'latency_p50_sec': percentile(trust_live_df['latency_sec'].dropna().tolist(), 0.50),
        'latency_p95_sec': percentile(trust_live_df['latency_sec'].dropna().tolist(), 0.95),
    })
    trust_live_summary_df = pd.DataFrame([metrics])
trust_live_summary_df


,n_valid,n_extreme,accuracy,precision,recall,f1,roc_auc,spearman_rho,high_label_mean,low_label_mean,tp,tn,fp,fn,requested,failure_rate,latency_p50_sec,latency_p95_sec
0,30,20,0.8,0.75,0.9,0.818182,0.775,0.391408,82.3,66.6,9,7,3,1,30,0.0,None,None


### C-1. 외부 검색 교차검증(캐시 참조)

서울대 레이블이 붙은 `_fc_title` 주장에 Google Search grounding을 적용한 오프라인 결과를 읽는다. API 호출은 이 노트북이 수행하지 않으며, 전체 가중치 탐색 수치와 5-fold out-of-fold 수치를 구분한다.


In [6]:
grounding_summary_df = pd.DataFrame()
if CONFIG['trust_grounding_summary_path'].exists():
    grounding_payload = json.loads(CONFIG['trust_grounding_summary_path'].read_text(encoding='utf-8'))
    cv = grounding_payload.get('five_fold_selection') or {}
    oof = cv.get('out_of_fold') or {}
    base_cv = cv.get('base_same_samples') or {}
    diagnostic = grounding_payload.get('blends', {}).get('alpha_0.2', {})
    grounding_summary_df = pd.DataFrame([{
        'n': oof.get('n'), 'base_spearman': base_cv.get('spearman_rho'),
        'oof_spearman': oof.get('spearman_rho'), 'base_extreme_auc': base_cv.get('extreme_auc'),
        'oof_extreme_auc': oof.get('extreme_auc'), 'mean_selected_alpha': cv.get('mean_alpha'),
        'diagnostic_alpha_0.2_spearman': diagnostic.get('spearman_rho'),
        'diagnostic_alpha_0.2_auc': diagnostic.get('extreme_auc'),
    }])
grounding_summary_df


,n,base_spearman,oof_spearman,base_extreme_auc,oof_extreme_auc,mean_selected_alpha,diagnostic_alpha_0.2_spearman,diagnostic_alpha_0.2_auc
0,30,0.391408,0.462104,0.775,0.8,0.28,0.552089,0.86


## D. 추천 — 합성 리플레이, 최근성, 다양성, 신뢰도 가드레일

DB 없이 768차원 합성 임베딩과 같은 카테고리 hard negative를 사용해 profile/category/latest/random 랭킹을 비교하고, 최근 기사 가중치·카테고리 60% 상한·저신뢰 후순위의 회귀 여부를 확인한다. 기본 합성 평가는 12건·K=3이며, 결과는 구현 검증이지 운영 품질의 증거는 아니다.


In [7]:
def basis(index: int, secondary: int | None = None, secondary_weight: float = 0.0) -> list[float]:
    vector = [0.0] * recommend.EMBED_DIM
    vector[index] = 1.0
    if secondary is not None:
        vector[secondary] = secondary_weight
    return vector

def rank_by_stable_random(sample: InteractionSample, _: dict[str, dict], candidates: list[str]) -> list[str]:
    material = f"{CONFIG['random_seed']}|{sample.subject_id}|{sample.positive_article_id}".encode('utf-8')
    seed = int.from_bytes(hashlib.sha256(material).digest()[:8], 'big')
    shuffled = list(candidates)
    random.Random(seed).shuffle(shuffled)
    return shuffled

def evaluate_with_stable_random(samples: list[InteractionSample], meta: dict[str, dict], k: int) -> dict[str, dict]:
    strategies = {**STRATEGIES, 'random': rank_by_stable_random}
    results = {name: {'hit': 0, 'mrr': 0.0, 'ndcg': 0.0, 'n': 0} for name in strategies}
    for sample in samples:
        candidates = [sample.positive_article_id, *sample.negative_article_ids]
        for name, rank_fn in strategies.items():
            ranked = rank_fn(sample, meta, candidates)
            rank = ranked.index(sample.positive_article_id) + 1
            bucket = results[name]
            bucket['n'] += 1
            bucket['hit'] += int(rank <= k)
            bucket['mrr'] += 1.0 / rank
            bucket['ndcg'] += 1.0 / math.log2(rank + 1) if rank <= k else 0.0
    return results

# 4개 의미 토픽을 2개 상위 카테고리에 배치한다. 같은 카테고리의 더 최신이지만
# 의미가 다른 기사를 hard negative로 넣어 profile과 category의 역할을 분리한다.
topic_category = {0: '경제', 1: '경제', 2: 'IT', 3: 'IT'}
synthetic_meta: dict[str, dict] = {}
synthetic_samples: list[InteractionSample] = []
for sample_no in range(12):
    topic = sample_no % 4
    sibling = topic + 1 if topic % 2 == 0 else topic - 1
    history_ids = [f's{sample_no}_h0', f's{sample_no}_h1']
    positive_id = f's{sample_no}_positive'
    hard_negative_id = f's{sample_no}_same_category_newer'
    synthetic_meta[history_ids[0]] = {'category': topic_category[topic], 'published_at': '2026-01-01', 'embedding': basis(topic)}
    synthetic_meta[history_ids[1]] = {'category': topic_category[topic], 'published_at': '2026-05-01', 'embedding': basis(topic, sibling, 0.05)}
    synthetic_meta[positive_id] = {'category': topic_category[topic], 'published_at': '2026-06-01', 'embedding': basis(topic, sibling, 0.10)}
    synthetic_meta[hard_negative_id] = {'category': topic_category[sibling], 'published_at': '2026-07-01', 'embedding': basis(sibling)}
    negative_ids = [hard_negative_id]
    for offset, negative_topic in enumerate([value for value in range(4) if value != topic], 1):
        negative_id = f's{sample_no}_negative_{offset}'
        synthetic_meta[negative_id] = {
            'category': topic_category[negative_topic], 'published_at': f'2026-07-{offset + 1:02d}',
            'embedding': basis(negative_topic),
        }
        negative_ids.append(negative_id)
    filler_id = f's{sample_no}_filler'
    synthetic_meta[filler_id] = {'category': '사회', 'published_at': '2026-07-10', 'embedding': basis(10)}
    negative_ids.append(filler_id)
    synthetic_samples.append(InteractionSample(f'synthetic_user_{sample_no}', history_ids, positive_id, negative_ids))

synthetic_k = min(3, max(1, CONFIG['top_k']))
synthetic_raw = evaluate_with_stable_random(synthetic_samples, synthetic_meta, synthetic_k)
synthetic_rec_df = pd.DataFrame([
    {
        'strategy': name, 'n': bucket['n'],
        f'HitRate@{synthetic_k}': bucket['hit'] / bucket['n'] if bucket['n'] else None,
        'MRR': bucket['mrr'] / bucket['n'] if bucket['n'] else None,
        f'NDCG@{synthetic_k}': bucket['ndcg'] / bucket['n'] if bucket['n'] else None,
    } for name, bucket in synthetic_raw.items()
])

recent_profile = recommend.build_profile_vector([
    {'embed_summary': basis(0)}, {'embed_summary': basis(1)},
])
ranking_records = [
    *[{'article_id': f'e{i}', 'category': '경제', 'trust_score': 80-i} for i in range(6)],
    *[{'article_id': f't{i}', 'category': 'IT', 'trust_score': 75-i} for i in range(4)],
]
diversified_top = recommend._diversify_by_category(ranking_records, limit=5)[:5]
guarded = recommend._apply_trust_guardrail([
    {'article_id': 'low', 'trust_score': 20}, {'article_id': 'high', 'trust_score': 85},
    {'article_id': 'unanalyzed', 'trust_score': 0},
])
counts = Counter(row['category'] for row in diversified_top)
recommend_unit_df = pd.DataFrame([
    {'test': 'profile_dimension_768', 'actual': len(recent_profile), 'expected': 768, 'pass': len(recent_profile) == 768},
    {'test': 'recent_item_has_more_weight', 'actual': recent_profile[1] > recent_profile[0], 'expected': True, 'pass': recent_profile[1] > recent_profile[0]},
    {'test': 'category_share_at_most_60pct', 'actual': max(counts.values()) / 5, 'expected': '<=0.6', 'pass': max(counts.values()) / 5 <= 0.6},
    {'test': 'low_trust_demoted', 'actual': guarded[-1]['article_id'], 'expected': 'low', 'pass': guarded[-1]['article_id'] == 'low'},
])
assert recommend_unit_df['pass'].all(), recommend_unit_df.loc[~recommend_unit_df['pass']]
display(synthetic_rec_df)
recommend_unit_df


,strategy,n,HitRate@3,MRR,NDCG@3
0,profile,12,1.000000,1.000000,1.000000
1,hybrid,12,1.000000,0.750000,0.815465
2,category,12,1.000000,0.333333,0.500000
3,latest,12,0.000000,0.166667,0.000000
4,random,12,0.416667,0.334722,0.271822


,test,actual,expected,pass
0,profile_dimension_768,768,768,True
1,recent_item_has_more_weight,True,True,True
2,category_share_at_most_60pct,0.6,<=0.6,True
3,low_trust_demoted,low,low,True


In [8]:
benchmark_history = [{'embed_summary': basis(i % 4)} for i in range(20)]
latencies = []
tracemalloc.start()
for _ in range(100):
    started = time.perf_counter()
    recommend.build_profile_vector(benchmark_history)
    latencies.append((time.perf_counter() - started) * 1000)
_, peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
recommend_efficiency_df = pd.DataFrame([{
    'operation': 'build_profile_vector(history=20, dim=768)', 'runs': len(latencies),
    'latency_p50_ms': percentile(latencies, 0.50), 'latency_p95_ms': percentile(latencies, 0.95),
    'peak_python_memory_mb': peak_bytes / 1024**2,
}])
recommend_efficiency_df


,operation,runs,latency_p50_ms,latency_p95_ms,peak_python_memory_mb
0,"build_profile_vector(history=20, dim=768)",100,11.9112,16.4922,0.163087


## E. 추천 — 선택적 실제 DB 오프라인 리플레이

`RUN_DB_RECOMMENDATION=1`일 때 같은 사용자/세션의 클릭 시점을 재생한다. 각 시점 이전 이력만 사용하며 실제 클릭 1개와 네거티브 후보를 랭킹한다. 최소 샘플 수가 작으면 수치를 결론으로 사용하지 않는다.


In [9]:
db_rec_error = None
db_rec_df = pd.DataFrame()
db_readiness_df = pd.DataFrame()
if CONFIG['run_db_recommendation']:
    if not os.getenv('DATABASE_URL'):
        db_rec_error = 'RUN_DB_RECOMMENDATION=1이지만 DATABASE_URL이 없습니다.'
    else:
        try:
            from backend.training.evaluate_offline import load_article_meta
            from backend.training.samples import load_samples_from_db
            samples = load_samples_from_db(history_size=20, negative_count=CONFIG['negative_count'])
            samples = [sample for sample in samples if len(sample.history_article_ids) >= CONFIG['min_history']]
            meta = load_article_meta()
            raw = evaluate_recommendation(samples, meta, CONFIG['top_k'])
            db_rec_df = pd.DataFrame([
                {
                    'strategy': name, 'n': bucket['n'],
                    f'HitRate@{CONFIG["top_k"]}': bucket['hit'] / bucket['n'] if bucket['n'] else None,
                    'MRR': bucket['mrr'] / bucket['n'] if bucket['n'] else None,
                    f'NDCG@{CONFIG["top_k"]}': bucket['ndcg'] / bucket['n'] if bucket['n'] else None,
                } for name, bucket in raw.items()
            ])
            embedded = sum(bool(item.get('embedding')) for item in meta.values())
            db_readiness_df = pd.DataFrame([{
                'replay_samples': len(samples), 'articles': len(meta), 'articles_with_embed_summary': embedded,
                'embedding_coverage': embedded / len(meta) if meta else 0.0,
                'enough_for_initial_conclusion': len(samples) >= 100,
            }])
        except Exception as exc:
            db_rec_error = f'{type(exc).__name__}: {exc}'
if db_rec_df.empty:
    db_rec_df = pd.DataFrame([{'status': db_rec_error or 'skipped_by_RUN_DB_RECOMMENDATION'}])
display(db_readiness_df if not db_readiness_df.empty else pd.DataFrame([{'status': db_rec_error or 'not_run'}]))
db_rec_df


,status
0,not_run


,status
0,skipped_by_RUN_DB_RECOMMENDATION


## F. 결과 보고서 저장

실행 메타데이터와 모든 결과표를 Markdown 하나와 CSV 묶음으로 저장한다. 보고서의 `skipped`는 실패가 아니라 해당 환경 스위치를 끈 상태다.


In [10]:
def markdown_table(frame: pd.DataFrame) -> str:
    if frame is None or frame.empty:
        return '_결과 없음_'
    safe = frame.copy().fillna('')
    columns = [str(column) for column in safe.columns]
    lines = ['| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join(['---'] * len(columns)) + ' |']
    for row in safe.astype(str).itertuples(index=False, name=None):
        lines.append('| ' + ' | '.join(value.replace('|', '\|').replace('\n', ' ') for value in row) + ' |')
    return '\n'.join(lines)

persona_rows = []
for persona_path in sorted((CONFIG['report_dir'].parent / 'optimization').glob('persona_benchmark_seed*.json')):
    payload = json.loads(persona_path.read_text(encoding='utf-8'))
    candidate = payload['baselines']['deployment_candidate']
    semantic = payload['baselines']['semantic_only']
    persona_rows.append({
        'seed': payload['seed'], 'validation_n': payload['validation_count'],
        'semantic_ndcg3': semantic['ndcg_at_3'], 'hybrid_ndcg3': candidate['ndcg_at_3'],
        'semantic_mrr': semantic['mrr'], 'hybrid_mrr': candidate['mrr'],
        'semantic_low_trust': semantic['low_trust_topk_rate'], 'hybrid_low_trust': candidate['low_trust_topk_rate'],
    })
persona_seed_df = pd.DataFrame(persona_rows)
persona_summary_df = pd.DataFrame() if persona_seed_df.empty else pd.DataFrame([{
    'seeds': len(persona_seed_df), 'validation_n_total': int(persona_seed_df['validation_n'].sum()),
    'semantic_ndcg3': persona_seed_df['semantic_ndcg3'].mean(), 'hybrid_ndcg3': persona_seed_df['hybrid_ndcg3'].mean(),
    'ndcg_relative_gain': persona_seed_df['hybrid_ndcg3'].mean() / persona_seed_df['semantic_ndcg3'].mean() - 1,
    'semantic_mrr': persona_seed_df['semantic_mrr'].mean(), 'hybrid_mrr': persona_seed_df['hybrid_mrr'].mean(),
    'mrr_relative_gain': persona_seed_df['hybrid_mrr'].mean() / persona_seed_df['semantic_mrr'].mean() - 1,
    'low_trust_relative_change': persona_seed_df['hybrid_low_trust'].mean() / persona_seed_df['semantic_low_trust'].mean() - 1,
}])

current_summary = (trust_pair_summary_df[trust_pair_summary_df.get('is_current', False) == True]
                   if 'is_current' in trust_pair_summary_df.columns else pd.DataFrame())
decision_df = pd.DataFrame([
    {
        'result_scope': '신뢰도 산식·경계 구현',
        'status': '사용 가능 — 구현 검증',
        'basis': f"단위검증 {int(trust_unit_df['pass'].sum())}/{len(trust_unit_df)} 통과",
    },
    {
        'result_scope': '신뢰도 모델 성능',
        'status': '조건부 사용 가능' if not trust_live_df.empty and len(trust_live_df) >= 30 else '보류',
        'basis': f'Live 표본 {len(trust_live_df)}건; 30건 미만 또는 미실행이면 성능 결론 금지',
    },
    {
        'result_scope': '추천 로직 회귀',
        'status': '사용 가능 — 합성 구현 검증',
        'basis': f"합성 표본 {int(synthetic_rec_df['n'].max())}건, 후처리 {int(recommend_unit_df['pass'].sum())}/{len(recommend_unit_df)} 통과",
    },
    {
        'result_scope': '운영 추천 품질',
        'status': '조건부 사용 가능' if not db_readiness_df.empty and int(db_readiness_df.iloc[0]['replay_samples']) >= 100 else '보류',
        'basis': '시간 순서를 보존한 DB 리플레이 100건 이상이 필요함',
    },
])

tables = {
    'decision': decision_df,
    'runtime': runtime_df,
    'trust_unit': trust_unit_df,
    'trust_pair_summary': trust_pair_summary_df,
    'trust_pair_detail': trust_pair_df,
    'trust_live_summary': trust_live_summary_df,
    'trust_grounding_summary': grounding_summary_df,
    'recommend_synthetic': synthetic_rec_df,
    'recommend_unit': recommend_unit_df,
    'recommend_efficiency': recommend_efficiency_df,
    'recommend_db_readiness': db_readiness_df,
    'recommend_db_replay': db_rec_df,
    'persona_seed_detail': persona_seed_df,
    'persona_summary': persona_summary_df,
}
for name, frame in tables.items():
    if frame is not None and not frame.empty:
        frame.to_csv(CONFIG['report_dir'] / f'{name}.csv', index=False, encoding='utf-8-sig')

interpretations = []
if worktree_dirty:
    interpretations.append(f'평가는 미커밋 변경이 있는 워크트리에서 실행됐다. 재현 시 commit뿐 아니라 evaluation_code_sha256={evaluation_code_sha256}을 함께 확인해야 한다.')
interpretations.append('현재 로컬 실행 환경의 Attention 경로는 ' + ('사용 가능' if encoder_inference.is_model_ready() else '비활성') + ' 상태다. 이 결과는 Render 준비 상태를 의미하지 않는다.')
if not current_summary.empty:
    current = current_summary.iloc[0]
    interpretations.append(f"현재 운영 가중치 {current['weights']}의 캐시 불변쌍 통과율은 {current['pass_rate']:.1%}이며 실패 사례는 {current['failed_cases']}다.")
    if trust_pair_summary_df['pass_rate'].nunique() == 1:
        interpretations.append('불변쌍 통과율만으로 가중치 우열을 정할 수 없어, SNU 팩트체크 상관과 AUC가 가장 높은 W2를 선택했다.')
if not trust_live_df.empty:
    interpretations.append(f"Live 신뢰도 평가 {len(trust_live_df)}건 중 실패율은 {trust_live_df['failed'].mean():.1%}다.")
    if 'source_mode' in trust_live_df.columns:
        interpretations.append('신뢰도 성능은 저장된 SNU Live criterion 결과를 현재 W2로 재가중한 값이며 추가 API 호출은 없었다.')
else:
    interpretations.append('Live 신뢰도 평가는 비활성 상태이므로 실제 모델 정확도·지연 결론은 보류한다.')
if not grounding_summary_df.empty:
    grounded = grounding_summary_df.iloc[0]
    interpretations.append(f"외부 검색 교차검증 5-fold OOF에서 Spearman은 {grounded['base_spearman']:.3f}→{grounded['oof_spearman']:.3f}, 극단 AUC는 {grounded['base_extreme_auc']:.3f}→{grounded['oof_extreme_auc']:.3f}로 개선됐다. 전체 30건 alpha=0.2 수치는 진단값이며 독립 검증값이 아니다.")
if not db_readiness_df.empty:
    n = int(db_readiness_df.iloc[0]['replay_samples'])
    interpretations.append(f'DB 리플레이 유효 샘플은 {n}건이다.' + (' 초기 비교가 가능하다.' if n >= 100 else ' 100건 미만이므로 방향성만 참고한다.'))
else:
    interpretations.append('DB 리플레이가 비활성 상태이므로 운영 추천 품질 결론은 보류한다.')
profile_mrr = float(synthetic_rec_df.loc[synthetic_rec_df['strategy'] == 'profile', 'MRR'].iloc[0])
category_mrr = float(synthetic_rec_df.loc[synthetic_rec_df['strategy'] == 'category', 'MRR'].iloc[0])
interpretations.append(f'합성 hard-negative 회귀에서 profile MRR={profile_mrr:.3f}, category MRR={category_mrr:.3f}이다. 이는 구현 경로 구분용이며 운영 성능 수치로 인용하지 않는다.')
if not persona_summary_df.empty:
    persona_result = persona_summary_df.iloc[0]
    interpretations.append(f"인간 직관 페르소나 5-seed에서 hybrid NDCG@3 상대 개선은 {persona_result['ndcg_relative_gain']:.1%}, MRR 개선은 {persona_result['mrr_relative_gain']:.1%}다.")
interpretations.append('DB 리플레이는 클릭 시점 이전 impression만 사용하고 30초 내 중복 positive 및 임베딩 없는 기사를 제거한 temporal sampler를 사용한다.')
interpretations.append('프로필 벡터 지연시간은 DB 검색·네트워크·Attention 추론을 제외한 로컬 microbenchmark다.')
interpretations.append('profile 단계는 raw embed_summary만 사용하며 raw chunk embedding 후보와 같은 공간에서 검색한다. Attention 경로의 learned_embedding은 별도 공간으로 분리한다.')

sections = [
    '# ET Core Function Experiment Report',
    f"- 실행 시각(UTC): {datetime.now(timezone.utc).isoformat()}",
    f"- Git commit: {git_value('rev-parse', '--short', 'HEAD')}",
    f"- Worktree dirty: {worktree_dirty}",
    f"- Evaluation code SHA-256: {evaluation_code_sha256}",
    f"- RUN_LIVE_TRUST: {CONFIG['run_live_trust']}",
    f"- RUN_DB_RECOMMENDATION: {CONFIG['run_db_recommendation']}",
    '## 결과 사용 판정\n' + markdown_table(decision_df),
    '## 핵심 해석\n' + '\n'.join(f'- {item}' for item in interpretations),
]
for title, key in [
    ('런타임 준비도', 'runtime'), ('신뢰도 산식 회귀검증', 'trust_unit'),
    ('신뢰도 불변쌍 가중치 비교', 'trust_pair_summary'), ('신뢰도 Live 평가', 'trust_live_summary'),
    ('외부 검색 교차검증', 'trust_grounding_summary'),
    ('추천 합성 랭킹', 'recommend_synthetic'), ('추천 후처리 회귀검증', 'recommend_unit'),
    ('추천 로컬 효율', 'recommend_efficiency'), ('추천 DB 준비도', 'recommend_db_readiness'),
    ('추천 DB 리플레이', 'recommend_db_replay'),
    ('인간 직관 페르소나 요약', 'persona_summary'), ('페르소나 seed 상세', 'persona_seed_detail'),
]:
    sections.append(f'## {title}\n' + markdown_table(tables[key]))

report_path = CONFIG['report_dir'] / 'Core_Function_experiment_report.md'
report_path.write_text('\n\n'.join(sections) + '\n', encoding='utf-8')
print(f'보고서 저장 완료: {report_path}')
print(f'CSV 저장 폴더: {CONFIG["report_dir"]}')
report_path


보고서 저장 완료: C:\Users\user\Desktop\ET_by_claude\FINAL_ANALYSIS\artifacts\core_function_experiment\Core_Function_experiment_report.md
CSV 저장 폴더: C:\Users\user\Desktop\ET_by_claude\FINAL_ANALYSIS\artifacts\core_function_experiment


WindowsPath('C:/Users/user/Desktop/ET_by_claude/FINAL_ANALYSIS/artifacts/core_function_experiment/Core_Function_experiment_report.md')